# 📘 Notebook 13: Meet a Real Warehouse

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 1 of 6 in this module · (13 of 18 overall)*

---

> 🧭 **How to use this notebook.** Run the cells in order, from top to bottom: click a cell and press **Shift+Enter**. After most cells there is a short paragraph headed **What you should see**: compare it with your own output before you move on. If your numbers differ, stop and find out why, because that is how data work is learned. The exercises at the end start easy and get harder: first **warm-up** questions, then **think it through** questions (what would you run, and why?), then **technical** tasks. Try each one before you open its solution.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Download a real data warehouse and take an **inventory** of its tables
- Tell **fact** tables from **dimension** tables, and state the **grain** of a fact table with evidence
- Measure **missing values** and decide which ones matter
- Check **referential integrity**: does every key in a fact table find its partner?
- Spot **personal data** and apply the principle of data minimisation before any analysis

> **Prerequisites:** none beyond basic Python. We use **pandas**, the standard Python library for tables, and explain every command the first time it appears. (A longer introduction to pandas is in *From Python to Deep Learning and Fine-Tuning*, Notebook 06. Keys, joins and the star schema are in Notebooks 01, 02 and 04 of this course.)
>
> 🔭 **Why this notebook matters:** real data always holds surprises. Before a single chart or model, a professional spends time simply *getting to know* the data: what is there, how it is shaped, what is missing, what does not join, and what should not be there at all. This step is unglamorous, and it is where most analytical mistakes are prevented. In this module we take one real warehouse through the whole journey: profile it (this notebook), build it, reshape it, explore it with Spark, learn from it, and measure what scale really means for it.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

You do not need to understand the code of the next cell. It downloads the data and defines one helper that we use all the time: **`load("TableName")`** reads one table into a pandas **DataFrame**, a table with named columns and numbered rows.

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

**What you should see:** `Tables:` followed by ten names, from `DimCurrency` to `FactInternetSales`. Each name is one CSV file in the folder `adventureworks_dw` (open the folder icon on the left of Colab to see them).

## 1. An inventory

### Intuition first
Imagine being handed the keys to a stranger's warehouse. You would not start by rearranging the shelves: you would walk the aisles and write down what is there. With data, the walk takes a few lines of code.

The table names follow a common convention. Names beginning with **Fact** hold **events** that are counted and summed: here, the sales made on the company's website. Names beginning with **Dim** (dimension) **describe** what the events are about: customers, products, dates, places.

First, one table on its own, to meet the basic pandas commands:

In [ ]:
products = load("DimProduct")

print("rows and columns:", products.shape)      # .shape = (number of rows, number of columns)
print("number of rows:  ", len(products))
products.head(3)                                 # .head(3) = the first three rows

**What you should see:** `rows and columns: (606, 35)`, `number of rows: 606`, and the first three products as a table. Scroll sideways: 35 columns, from `ProductKey` to the product's names in English, Spanish and French, its colour, cost, price and much more.

`products.shape` and `len(products)` are the first two things to ask of any table. `.head()` shows a few rows, so you can see what the values look like.

Now the same for every table at once. The loop below loads each table, writes down its kind, rows, columns and the memory it takes, and collects the answers in a new small DataFrame.

In [ ]:
rows = []
for name in sorted(f[:-4] for f in os.listdir(DATA)):          # every file name without ".csv"
    t = load(name)
    rows.append({"table": name, "kind": "fact" if name.startswith("Fact") else "dimension",
                 "rows": len(t), "columns": t.shape[1],
                 "memory_MB": round(t.memory_usage(deep=True).sum() / 1e6, 1)})
inventory = pd.DataFrame(rows).sort_values("rows", ascending=False)
inventory

**What you should see:** ten rows, biggest first. One fact table, `FactInternetSales`, with 60,398 rows, 26 columns and about 29 MB in memory; then the customers (18,484), the dates (3,652), places (655), products (606), and five small dimensions down to 4 product categories.

In [ ]:
print("Rows in all tables:", inventory["rows"].sum())
print("Memory in pandas:   ", round(inventory["memory_MB"].sum()), "MB")

**What you should see:** `Rows in all tables: 83968` and about 58 MB.

`inventory["rows"]` picks one column; `.sum()` adds it up. Fewer than a hundred thousand rows and a few tens of megabytes: that is **small**. It fits many times over in the memory of any laptop. Keep the number in mind. A recurring question of this module is what would change if it were a thousand times larger, and the honest answer is not always "use Spark". Notebook 18 measures it.

## 2. The fact table and its grain

### Intuition first
The **grain** of a fact table is the answer to *"what is one row?"*. One sale? One order? One product in an order? One day of a shop? It is the most important sentence in a warehouse's documentation, because every sum and every count depends on it. Count the rows of a table whose grain you have misunderstood, and you will report the wrong number with complete confidence.

Look at the main fact table first.

In [ ]:
sales = load("FactInternetSales")
sales.head()

**What you should see:** five rows and 26 columns. Every row has keys that point to other tables (`ProductKey`, `CustomerKey`, `OrderDateKey` and more), an order number such as `SO43697` with a line number, and money columns such as `UnitPrice` and `SalesAmount`.

The column names suggest a hypothesis: one row is **one line of one sales order** (an order with three different products would be three rows). A hypothesis about grain can be **tested**. If it is true, the pair (`SalesOrderNumber`, `SalesOrderLineNumber`) never repeats.

Two new commands: `.drop_duplicates()` removes repeated rows, so the number of rows left is the number of *different* pairs. `.nunique()` counts the different values in one column.

In [ ]:
pairs = sales[["SalesOrderNumber", "SalesOrderLineNumber"]].drop_duplicates()
print("Rows:                  ", len(sales))
print("Distinct (order, line):", len(pairs))
print("Distinct orders:       ", sales["SalesOrderNumber"].nunique())
print("Distinct customers:    ", sales["CustomerKey"].nunique())
print("Distinct products:     ", sales["ProductKey"].nunique())

**What you should see:** 60,398 rows and 60,398 distinct pairs; 27,659 orders; 18,484 customers; 158 products.

The two first numbers are equal, so the pair never repeats: **the hypothesis holds**, and the pair can serve as the table's key. Had the second number been smaller, some pair would appear twice and the grain would be something else. Also: 60,398 lines from 27,659 orders means a little over two lines per order, and only 158 of the 606 products were ever sold online.

Now the **measures**, the numbers that will be summed. `.describe()` gives a quick summary of each numerical column: count, mean, standard deviation, minimum, quartiles and maximum.

In [ ]:
measures = ["OrderQuantity", "UnitPrice", "SalesAmount", "TotalProductCost", "TaxAmt", "Freight"]
sales[measures].describe().round(2)

**What you should see:** a table with 8 rows of statistics and 6 columns. Look at two things:

* `OrderQuantity` has minimum 1, maximum 1 and standard deviation 0: in this shop **every line is a single item**, so counting lines and summing quantities give the same answer.
* `UnitPrice` has a median (the `50%` row) of 29.99 but a maximum of 3,578.27 and a mean of about 486: a few expensive bicycles among many cheap accessories. When the mean is far above the median, a handful of large values pull it up. Remember this in Notebook 16.

## 3. Missing values

### Intuition first
An empty cell can mean three different things:

* *the value does not apply* (a customer with no middle name);
* *the value was never recorded* (a lost delivery note);
* *the value exists but the export lost it*.

Code cannot tell them apart. It can only **find** them; then a person must **decide**.

How the function below works: `df.isna()` gives a table of `True` (missing) and `False`; `.mean()` of True/False values is the share of `True`, because Python counts True as 1 and False as 0; times 100 it is a percentage. We keep only the columns with something missing.

In [ ]:
def missing_report(df):
    share = df.isna().mean().mul(100).round(1)          # % of missing values in each column
    return share[share > 0].sort_values(ascending=False)

missing_report(sales)

**What you should see:** two columns, `CarrierTrackingNumber` and `CustomerPONumber`, both **100%** missing. They carry no information at all and can be dropped.

In [ ]:
customers = load("DimCustomer")
missing_report(customers)

**What you should see:** four columns. `Suffix` 100%, `Title` 99.5%, `AddressLine2` 98.3% and `MiddleName` 42.4%.

These are all values that *do not apply* to most people: few people have a suffix (such as *Jr.*) or a second address line. None of them is a problem for analysis, and none of them would be used in it anyway. The point is that each one is now a **decision**, written down, and not an accident.

## 4. Do the keys join? Referential integrity

### Intuition first
A fact row points to its dimensions through keys: `CustomerKey = 11000` means *"look up customer 11000 in `DimCustomer`"*. If that customer is not there, every join between sales and customers will silently **drop** the sale. The total will be wrong and nothing will warn you. So we check, for each key, which share of fact rows finds a partner.

The new command is `.isin(...)`: for each value of a column it answers `True` if the value appears in another list. The share of `True` is again `.mean()`.

In [ ]:
def key_match(fact, fact_col, dim_name, dim_col=None):
    dim = load(dim_name)
    share = fact[fact_col].isin(dim[dim_col or fact_col]).mean() * 100
    print(f"{fact_col:>18} -> {dim_name:<18} {share:6.2f} % of rows have a partner")
    return share

for col, dim, dcol in [("CustomerKey", "DimCustomer", None), ("ProductKey", "DimProduct", None),
                       ("SalesTerritoryKey", "DimSalesTerritory", None), ("PromotionKey", "DimPromotion", None),
                       ("CurrencyKey", "DimCurrency", None), ("OrderDateKey", "DimDate", "DateKey"),
                       ("ShipDateKey", "DimDate", "DateKey"), ("DueDateKey", "DimDate", "DateKey")]:
    key_match(sales, col, dim, dcol)

**What you should see:** eight lines, every one with **100.00 %**.

Every key finds its partner. That is what a well-built warehouse looks like, and it is worth checking every time, because a copy of a warehouse is not always as well built as the original. (If you are given another export of these tables, run this cell on it first.)

Notice the last three lines: the same `DimDate` table answers for three different dates of an order (ordered, shipped, due). A dimension used in several roles is called a **role-playing dimension**. We meet it again in Notebook 14.

The dimensions point to each other too. A product belongs to a subcategory, which belongs to a category:

In [ ]:
subcats = load("DimProductSubcategory")
print("Products:", len(products))
print("Products without a subcategory:", products["ProductSubcategoryKey"].isna().sum())
print("...of which ever sold online:", products.loc[products["ProductSubcategoryKey"].isna(), "ProductKey"]
      .isin(sales["ProductKey"]).sum())

**What you should see:** 606 products, 209 without a subcategory, and **0** of those ever sold online.

`products.loc[condition, "ProductKey"]` reads as *"the ProductKey of the products where the condition is true"*. A third of the products have no subcategory. They are parts (bolts, washers, frames for assembly) that the company keeps in its product list but never sells on the website, so they never appear in a sale. Missing, but harmless, and now we know why.

## 5. A first look at the business

The dates are stored as text such as `2010-12-29 00:00:00.000`. `pd.to_datetime` turns them into real dates, which know their year, month and day. Then `.groupby(year)` makes one group per year, and `.agg(["count", "sum"])` counts the lines and adds up the revenue of each group.

In [ ]:
sales["OrderDate"] = pd.to_datetime(sales["OrderDate"])
print("First order:", sales["OrderDate"].min().date(), "  Last order:", sales["OrderDate"].max().date())
sales.groupby(sales["OrderDate"].dt.year)["SalesAmount"].agg(["count", "sum"]).round(0)

**What you should see:** the first order on 2010-12-29 and the last on 2014-01-28. Then five years: 14 lines in 2010, 2,216 in 2011, 3,397 in 2012, 52,801 in 2013 and 1,970 in 2014, with revenue of about 7.1, 5.8 and 16.4 million in the three full years.

Two warnings are hidden in that small table:

* The first and last years are **not complete**: 2010 has three days and 2014 four weeks. A comparison of 2013 with 2014 would be meaningless.
* Between 2012 and 2013 the number of order lines grows about fifteen times while revenue grows less than three times: the shop must have started selling something new and **cheap**. A good first look raises questions like these. Exercise 6 answers this one.

In [ ]:
per_customer = sales.groupby("CustomerKey")["SalesOrderNumber"].nunique()   # orders per customer
per_customer.value_counts().sort_index().head(8)

**What you should see:** 11,619 customers with one order, 5,454 with two, 1,166 with three, and a quickly shrinking tail, down to 14 customers with 16 orders each.

Most customers ordered once. The few who ordered many times are worth a look before any average is computed, because they can pull it.

## 6. Personal data: keep only what the analysis needs

### Intuition first
The customer table holds names, e-mail addresses, phone numbers, street addresses and dates of birth. They are invented here, but in a real warehouse they would belong to real people, and data protection law (in Europe, the GDPR) says that personal data must be **adequate, relevant and limited to what is necessary**. Analysis of sales needs an age band, a region and an income band. It does not need anyone's name or telephone number.

The cleanest moment to apply this is now, before the data spreads into notebooks, charts and copies. `.drop(columns=[...])` returns the table without the listed columns.

In [ ]:
identifying = ["Title", "FirstName", "MiddleName", "LastName", "Suffix", "EmailAddress", "Phone",
               "AddressLine1", "AddressLine2", "CustomerAlternateKey"]
analysis_customers = customers.drop(columns=identifying)
print("Columns kept:", list(analysis_customers.columns))

**What you should see:** 19 columns kept, from `CustomerKey` and `GeographyKey` to `CommuteDistance`. No names, e-mails, phones or street addresses.

The `CustomerKey` stays: it is a meaningless number that links a customer's sales without saying who they are. That is a **pseudonym**. The date of birth is still personal; Exercise 9 replaces it with something less revealing.

## ⚠️ Common pitfalls

* **Counting rows before you know the grain.** Here a row is an order *line*: `len(sales)` counts lines, not orders and not customers.
* **Treating every missing value as a problem**, or as no problem. Find out *why* it is missing first.
* **Joining without checking the keys.** An unmatched key does not raise an error; the row just disappears from the join.
* **Comparing incomplete periods**, such as the four weeks of 2014 against the whole of 2013.
* **Carrying personal data "just in case".** If the analysis does not need it, drop it at the start.

## 7. Where we stand

| The question | How we answered it |
|---|---|
| What is in the warehouse? | an inventory: tables, rows, columns, memory |
| What is one row of the fact table? | a grain hypothesis, tested on the key |
| What is missing, and does it matter? | a missing-value report, read column by column |
| Does every key join? | a match rate for every foreign key, and for links between dimensions |
| What should not be here at all? | personal data, removed before the analysis starts |

None of this produced a chart or a model, and all of it was necessary. The next notebook takes the profiled tables and builds them into a database of our own, with the keys and rules that make the checks of this notebook permanent.

---
## ✏️ Exercises

## Level 1: warm-up

Each answer needs one or two of the commands of this notebook: `len`, `.shape`, `.head()`, `.nunique()`, `missing_report`.

### Exercise 1 (Sizes)
How many rows and columns do `DimDate` and `DimGeography` have? How many years of days does `DimDate` cover (rows divided by 365)?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
dates = load("DimDate")
geo = load("DimGeography")
print("DimDate:     ", dates.shape)
print("DimGeography:", geo.shape)
print("years of days:", round(len(dates) / 365, 1))
```

*DimDate has 3,652 rows and 19 columns, exactly ten years of days; DimGeography has 655 rows and 11 columns. A date dimension usually covers more days than the sales, so that future sales find their date.*

</details>

### Exercise 2 (Missing values in the products)
Run `missing_report` on `products`. Which three columns are missing most often? Pick one and say whether you think the value *does not apply* or *was lost*.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
missing_report(products).head(3)
```

*EndDate (67%), SizeUnitMeasureCode (58%) and WeightUnitMeasureCode (54%). All three are values that do not apply: EndDate is empty while a product version is still current, and many parts and accessories have no size or weight to measure. Missing for a reason, not lost.*

</details>

### Exercise 3 (Customers of one year)
How many different customers placed an order in 2013? (Hint: keep the rows of 2013 with `sales[sales["OrderDate"].dt.year == 2013]`, then use `.nunique()`.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
in_2013 = sales[sales["OrderDate"].dt.year == 2013]
print("customers in 2013:", in_2013["CustomerKey"].nunique())
```

*17,429 customers ordered in 2013, almost all of the 18,484 in the table. Filtering rows with a condition in square brackets is one of the most used commands in pandas.*

</details>

## Level 2: think it through

These questions are about **reasoning**. Before you write code, decide what you would need to see to answer the question. Then write the code that shows it.

### Exercise 4 (The grain of another table)
Dimension tables have a grain too. What is one row of `DimGeography`? **What would you count to find out?** Test whether `GeographyKey` is unique, and whether the pair (`City`, `PostalCode`) is.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
geo = load("DimGeography")
print("Rows:", len(geo))
print("Distinct GeographyKey:", geo["GeographyKey"].nunique())
print("Distinct (City, PostalCode):", len(geo[["City", "PostalCode"]].drop_duplicates()))
```

*655 rows and 655 distinct keys, so one row is one place. The pair (City, PostalCode) has 646 distinct values: very nearly unique, but not quite, so it could not serve as the key.*

</details>

### Exercise 5 (Was anything lost or doubled?)
Someone sends you a new copy of the sales table and says it is the same as yours. The cell below makes such a copy, with a fault hidden in it. **Which three numbers would you compare between the two copies to find out whether rows were lost or doubled?** Compute them for both, and say what happened.

```python
copy = pd.concat([sales.drop(index=range(1000, 1300)), sales.head(50)])
```

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
copy = pd.concat([sales.drop(index=range(1000, 1300)), sales.head(50)])

for label, t in [("original", sales), ("copy", copy)]:
    pairs = len(t[["SalesOrderNumber", "SalesOrderLineNumber"]].drop_duplicates())
    print(f"{label:<9} rows {len(t):>6}   distinct (order, line) {pairs:>6}   revenue {t['SalesAmount'].sum():>12,.2f}")
```

*The copy has 250 fewer rows, but 300 fewer distinct pairs: 300 rows were lost and 50 rows appear twice. Revenue is about 794,500 lower. The row count alone would have reported only 250 missing rows; the distinct key count and a total together reveal both faults.*

</details>

### Exercise 6 (What happened in 2013?)
Section 5 showed many more order lines in 2013 but much less growth in revenue. **What table would explain it?** Join the sales to the product, subcategory and category tables and show the number of order lines and the revenue **per category and year**. Which categories did the shop begin selling, and when?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
cats = load("DimProductCategory")
lines = (sales.merge(products[["ProductKey", "ProductSubcategoryKey"]], on="ProductKey")
              .merge(subcats[["ProductSubcategoryKey", "ProductCategoryKey"]], on="ProductSubcategoryKey")
              .merge(cats[["ProductCategoryKey", "EnglishProductCategoryName"]], on="ProductCategoryKey"))
lines.pivot_table(index=lines["OrderDate"].dt.year, columns="EnglishProductCategoryName",
                  values="SalesAmount", aggfunc=["count", "sum"]).round(0)
```

*Until 2011 the website sold only bicycles. Accessories and clothing appear in 2012, in small numbers, and in 2013 they outnumber the bicycle lines several times over (34,409 accessory lines against 9,706 bicycle lines) while adding much less revenue. Any trend computed over the whole period mixes two different shops.*

</details>

## Level 3: technical

Longer tasks: write a small reusable function, and transform a column.

### Exercise 7 (A reusable integrity check)
Write a function `integrity(fact, links)` that takes the fact table and a list of `(fact_column, dimension_name, dimension_column)` and returns a DataFrame with one row per link and the columns `link` and `match_percent`. Run it on the links of Section 4.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def integrity(fact, links):
    out = []
    for col, dim_name, dim_col in links:
        dim = load(dim_name)
        out.append({"link": f"{col} -> {dim_name}.{dim_col or col}",
                    "match_percent": round(fact[col].isin(dim[dim_col or col]).mean() * 100, 2)})
    return pd.DataFrame(out)

integrity(sales, [("CustomerKey", "DimCustomer", None), ("ProductKey", "DimProduct", None),
                  ("OrderDateKey", "DimDate", "DateKey")])
```

*Every link shows 100.0. A table is a better record than printed lines: it can be saved, compared between two copies of the data, and pasted into a report.*

</details>

### Exercise 8 (Unmatched keys, found)
Make a damaged copy of the sales in which 20 rows point to customers that do not exist: `bad = sales.copy(); bad.loc[:19, "CustomerKey"] = 999999`. Run `key_match` on it, then **list** the unmatched rows with `~` ("not") and `.isin`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
bad = sales.copy()
bad.loc[:19, "CustomerKey"] = 999999
key_match(bad, "CustomerKey", "DimCustomer")
unmatched = bad[~bad["CustomerKey"].isin(customers["CustomerKey"])]
print("unmatched rows:", len(unmatched))
unmatched[["SalesOrderNumber", "SalesOrderLineNumber", "CustomerKey"]].head()
```

*key_match reports 99.97%, a drop that is easy to overlook in a percentage, yet those 20 sales would vanish from every join with customers. Listing the unmatched rows makes them visible and gives the evidence for a data quality log.*

</details>

### Exercise 9 (Age instead of a birth date, a little harder)
Replace `BirthDate` in `analysis_customers` with an **age band** (`under 35`, `35-49`, `50-64`, `65 and over`), computed at the date of the last order in the data. Then drop `BirthDate` and show how many customers fall in each band.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
reference = sales["OrderDate"].max()
born = pd.to_datetime(analysis_customers["BirthDate"])
age = (reference - born).dt.days // 365
analysis_customers["AgeBand"] = pd.cut(age, bins=[0, 34, 49, 64, 200],
                                       labels=["under 35", "35-49", "50-64", "65 and over"])
analysis_customers = analysis_customers.drop(columns=["BirthDate"])
analysis_customers["AgeBand"].value_counts().sort_index()
```

*4,317 customers under 35, 8,541 aged 35-49, 4,451 aged 50-64 and 1,175 aged 65 and over. Four bands are enough for most analyses and reveal far less about a person than an exact date of birth. Computing the age at a fixed reference date, and not today, keeps the result reproducible.*

</details>

## 🔑 Key takeaways

- **Profile before you analyse**: an inventory, the grain, the missing values, the keys, the personal data.
- The first questions to ask of any table: `.shape`, `.head()`, and what one row means.
- The **grain** of a fact table is a hypothesis you can test: its key must be unique.
- **Missing values** need a decision, not just a count: does the value not apply, or was it lost?
- Check **referential integrity** for every foreign key. An unmatched key makes joins drop rows without warning.
- To compare two copies of a table, compare row counts, distinct keys and a total: together they reveal lost and doubled rows.
- Apply **data minimisation** at the start: keep pseudonymous keys and coarse bands, drop names and contact details.

**Next:** *Notebook 14: Building the Warehouse*, where these tables become a database of our own, with keys and rules, views and indexes, and the queries an analyst would ask of it.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*